# Run DINGO over the sample

Fits and samples every staged source, start to finish, on the GPU. Nothing needs
a terminal and nothing shells out to another interpreter: this notebook runs on
the **`dingo`** kernel, which is Python 3.12 with torch and CUDA, so the package
is imported directly.

Edit the cell below and run the notebook top to bottom.

**Timing.** The batched likelihood is ~17x faster on the GPU, and its cost is
nearly flat in walker count (256 walkers cost 28% more per step than 32, not
8x). A 256-walker, 10000-step chain is about **23 minutes**; the seven
Paschen-alpha sources take roughly **3 hours** in sequence. On CPU the same work
is about a week.

## Settings — this is the only cell you need to edit

In [ ]:
from dingo import catalogs

# Sources. catalogs.SAMPLE_IDS is the seven Paschen-alpha sources that have data
# staged; ID6084 is the H-alpha one at z=5.7 and is not in that list.
SOURCES       = catalogs.SAMPLE_IDS          # or e.g. [15665, 21636]

SKIP_EXISTING = True     # skip a source that already has a chain on disk
MAKE_PLOTS    = True     # traces, corner and the R/C summary, per source
RUN_NULL_TEST = True     # rotation vs no-rotation likelihood ratio (adds ~1 min)

# Chain. None means "use whatever the config asks for".
#
# init: use 'adam'. 'prior' sounds like the right setting for exploring and is
# not: emcee's stretch move proposes along the line between two walkers, so when
# they start scattered across the full prior volume in 9-10 dimensions those
# lines almost never land in the narrow likelihood peak and the sampler freezes.
# Measured on ID3700, 512 walkers, only init changed:
#
#     init=prior   acceptance 0.025   walkers moved on  2.9% of steps
#     init=adam    acceptance 0.145   walkers moved on 14.9% of steps
#
# A first attempt at the sample with init='prior' produced three chains whose
# walkers moved on 0.1% of steps -- prior samples wearing a posterior's clothes,
# with V_rot = 2100 +- 400. They are in results/bad_initprior/.
#
# Note walkers do NOT improve the n/tau convergence gate -- that is nsteps/tau.
# More walkers buy more samples and better mode coverage, not a passing gate.
NWALKERS = 512
NSTEPS   = 10000
NBURN    = 2000
INIT     = 'adam'        # 'prior' to explore, 'adam' to polish a known solution

## Setup

In [ ]:
import os, sys, time, warnings, logging, traceback
import numpy as np
import torch

# The grism trace/dispersion tables. Not bundled with the package.
os.environ.setdefault('GRISM_CAL_DIR', '/home/aakavan/kinematics/grism_cal')

from dingo import fitting, plot, run as dingo_run

REPO = os.path.abspath(os.path.join(os.path.dirname(fitting.__file__), '..'))
DATA = os.path.join(REPO, 'data')
os.chdir(DATA)          # configs use paths relative to data/

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
for noisy in ('matplotlib', 'PIL', 'dingo.grism', 'dingo.utils'):
    logging.getLogger(noisy).setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GB)')
else:
    print('WARNING: no GPU visible. This will be ~20x slower, and roughly a week '
          'for the whole sample. Check that CUDA_VISIBLE_DEVICES is not set to "".')
print(f'repo   : {REPO}')
print(f'sources: {len(SOURCES)} -> {SOURCES}')

## Run

One source at a time. A failure is caught and recorded rather than stopping the
sample, so a bad source does not cost you the rest of the run.

Each source does: Adam fit -> `fit_report` (railed parameters, convergence) ->
freeze the noise model -> sample -> save -> convergence diagnostics.

In [ ]:
def run_one(sid):
    '''Fit and sample one source. Returns a row of results.'''
    tag   = str(sid) if str(sid).startswith('ID') else f'ID{sid}'
    cfg   = f'{tag}/config_kinematics.yaml'
    chain = dingo_run.chain_path(DATA, tag)
    row   = {'ID': tag, 'status': '', 'minutes': np.nan}

    f = fitting.KinematicsFitter(config_path=cfg, device=DEVICE)

    # Notebook settings override the config where they are not None.
    mc = f.config.setdefault('mcmc', {})
    for key, val in (('nwalkers', NWALKERS), ('nsteps', NSTEPS),
                     ('nburn', NBURN), ('init', INIT)):
        if val is not None:
            mc[key] = val

    f.fit_all()
    rep = f.fit_report()                      # report-only; does not raise
    row['adam_ok'] = rep['ok']
    row['railed']  = ','.join(sorted({p.split(':')[1].split(' is')[0].strip()
                                      for p in rep['problems'] if 'railed' in p})) or '-'

    f.fit_MCMC()
    os.makedirs(os.path.dirname(chain), exist_ok=True)
    f.save_mcmc(chain)

    d = f.get_mcmc_diagnostics()
    row['acceptance'] = d['acceptance_fraction'][0]
    row['worst_n_tau'] = min(d['n_steps_over_tau'].values())
    row['converged']   = row['worst_n_tau'] > 50

    res = f.get_mcmc_results()
    for name, short in (('velocity.V_sini', 'V_sini'), ('velocity.V_rot', 'V_rot'),
                        ('velocity.R_v', 'R_v'), ('velocity.inc_v', 'inc_v')):
        if name in res:
            lo, med, hi = res[name]
            row[short] = med
            row[short + '_err'] = 0.5*(hi - lo)

    if RUN_NULL_TEST:
        f.set_params_to_mcmc('map')
        null = f.compare_null_model()
        row['rotation_sigma'] = null['sigma_equiv']

    return f, row, chain


results, fitters = [], {}
t_all = time.time()

for sid in SOURCES:
    tag = str(sid) if str(sid).startswith('ID') else f'ID{sid}'
    if SKIP_EXISTING and os.path.exists(dingo_run.chain_path(DATA, tag)):
        print(f'{tag}: chain already on disk, skipping')
        results.append({'ID': tag, 'status': 'skipped'})
        continue

    print(f'\n{"="*70}\n{tag}\n{"="*70}')
    t0 = time.time()
    try:
        f, row, chain = run_one(sid)
        row['status'], row['minutes'] = 'ok', (time.time() - t0)/60
        fitters[tag] = f
        print(f'{tag}: done in {row["minutes"]:.1f} min -> {chain}')
    except Exception as exc:
        row = {'ID': tag, 'status': f'FAILED {type(exc).__name__}',
               'minutes': (time.time() - t0)/60}
        print(f'{tag}: FAILED after {row["minutes"]:.1f} min')
        traceback.print_exc(limit=3)
    results.append(row)

print(f'\n{"="*70}\ntotal {(time.time()-t_all)/60:.1f} min')

## Results

In [ ]:
import pandas as pd
df = pd.DataFrame(results).set_index('ID')
cols = [c for c in ('status','minutes','V_sini','V_sini_err','V_rot','V_rot_err',
                    'R_v','inc_v','acceptance','worst_n_tau','converged',
                    'rotation_sigma','adam_ok','railed') if c in df.columns]
display(df[cols].round(3))

bad = df[df.get('converged', True) == False]
if len(bad):
    print(f'\n{len(bad)} source(s) below n/tau = 50 -- too short to quote. '
          f'Raise NSTEPS and re-run; the chain resumes from its checkpoint.')
    print('  ' + ', '.join(bad.index))

## Figures

Traces and a corner plot per source, from the chains just written. Re-runnable
on its own: it reads the saved chains rather than depending on the loop above.

In [ ]:
if MAKE_PLOTS:
    import matplotlib.pyplot as plt
    for sid in SOURCES:
        tag = str(sid) if str(sid).startswith('ID') else f'ID{sid}'
        path = dingo_run.chain_path(DATA, tag)
        if not os.path.exists(path):
            continue
        d = fitting.BaseFitter.load_mcmc(path)
        names = [str(n) for n in d['param_names']]
        raw, nburn = np.asarray(d['chain']), int(d['nburn'])
        flat = raw[nburn:].reshape(-1, raw.shape[-1])

        print(f'\n{tag}: {flat.shape[0]:,} post-burn-in samples, {len(names)} parameters')
        plot.plot_mcmc_chains(raw, names, nburn=nburn, show=True)
        plot.plot_mcmc_corner(flat, names, show=True)
        plt.close('all')

## Notes

**Where things land.** Chains go to `results/ID<id>_chain.npz`. A checkpoint is
written every `checkpoint_every` steps, so an interrupted run resumes rather
than restarting — re-running this notebook picks up where it stopped.

**If a source is slow.** Check that `vectorize` is on. The batched path
evaluates every walker in one call; the scalar path does one per call and is
roughly 100x slower on a GPU. The default now follows the device, so this
should not happen, but an explicit `mcmc.vectorize: false` in a config would
still do it.

**If a source fails to start.** `fit_MCMC` refuses when the Adam fit lands
outside the prior support, and names the parameter. That usually means the fit
railed — `fit_report` will have said so — rather than that the prior is wrong.

**Error bars.** The posterior widths are known to be too small; see
`docs/validation.md`. The centres are sound.